# Exam Paper vs Module Matching - Google Colab

This notebook trains a Bloom taxonomy classifier from a dataset stored in **Google Drive**, saves the trained model as **`bloom_svm_model.h5`**, and provides a simple **two-input interface**:

- **Upload Module**
- **Upload Paper**

The system then:
1. extracts text from PDF / DOCX / TXT / image files,
2. predicts the Bloom level for each detected paper question,
3. checks whether the paper matches the module,
4. shows missing module items when coverage is low.


In [ ]:
# Install dependencies in Colab
!pip -q install pdfplumber python-docx pytesseract Pillow gradio h5py scikit-learn pandas numpy
!apt-get -qq update
!apt-get -qq install -y tesseract-ocr poppler-utils


In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


## Configuration

Set the dataset path to the CSV in your Google Drive.

Example:
`/content/drive/MyDrive/blooms_taxonomy_dataset.csv`


In [ ]:
DATASET_PATH = '/content/drive/MyDrive/blooms_taxonomy_dataset.csv'
MODEL_PATH = '/content/bloom_svm_model.h5'


In [ ]:
import io
import os
import re
import json
import pickle
from difflib import SequenceMatcher

import h5py
import numpy as np
import pandas as pd
import pdfplumber
import pytesseract
from PIL import Image
from docx import Document

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, classification_report


In [ ]:
def save_pipeline_to_h5(pipeline, label_classes, accuracy, output_path):
    with h5py.File(output_path, 'w') as h5f:
        model_bytes = pickle.dumps(pipeline)
        h5f.create_dataset('model_pickle', data=np.void(model_bytes))
        h5f.create_dataset('label_classes', data=json.dumps(list(label_classes)).encode('utf-8'))
        h5f.create_dataset('accuracy', data=accuracy)


def load_pipeline_from_h5(model_path):
    with h5py.File(model_path, 'r') as h5f:
        model_bytes = bytes(h5f['model_pickle'][()])
        pipeline = pickle.loads(model_bytes)
        label_classes = json.loads(h5f['label_classes'][()].decode('utf-8'))
        accuracy = float(h5f['accuracy'][()])
    return pipeline, label_classes, accuracy


## Train the model and create the `.h5` file

In [ ]:
df = pd.read_csv(DATASET_PATH)

required_cols = {'Questions', 'Category'}
if not required_cols.issubset(df.columns):
    raise ValueError(f'Dataset must contain columns: {required_cols}')

df = df.dropna(subset=['Questions', 'Category']).copy()
df['Questions'] = df['Questions'].astype(str).str.strip()
df['Category'] = df['Category'].astype(str).str.strip()
df = df[(df['Questions'] != '') & (df['Category'] != '')]

X = df['Questions']
y = df['Category']

x_train, x_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(
        lowercase=True,
        stop_words='english',
        ngram_range=(1, 2),
        max_features=20000
    )),
    ('clf', LinearSVC())
])

pipeline.fit(x_train, y_train)

y_pred = pipeline.predict(x_test)
accuracy = accuracy_score(y_test, y_pred)

print(f'Accuracy: {accuracy:.4f}')
print(classification_report(y_test, y_pred))

save_pipeline_to_h5(
    pipeline=pipeline,
    label_classes=sorted(y.unique()),
    accuracy=accuracy,
    output_path=MODEL_PATH
)

print(f'Model saved to: {MODEL_PATH}')


## File reading and text extraction

In [ ]:
ALLOWED_EXTENSIONS = {'txt', 'pdf', 'docx', 'png', 'jpg', 'jpeg'}


def get_extension_from_name(filename):
    filename = str(filename or '')
    if '.' not in filename:
        return ''
    return filename.rsplit('.', 1)[-1].lower()


def validate_extension_only(filename):
    ext = get_extension_from_name(filename)
    return ext in ALLOWED_EXTENSIONS


def read_txt_bytes(file_bytes):
    return file_bytes.decode('utf-8', errors='ignore')


def read_docx_bytes(file_bytes):
    buffer = io.BytesIO(file_bytes)
    doc = Document(buffer)
    return '\n'.join([p.text for p in doc.paragraphs if p.text.strip()])


def read_pdf_bytes(file_bytes):
    buffer = io.BytesIO(file_bytes)
    text_parts = []
    with pdfplumber.open(buffer) as pdf:
        for page in pdf.pages:
            page_text = page.extract_text()
            if page_text and page_text.strip():
                text_parts.append(page_text)
            else:
                image = page.to_image(resolution=300).original
                text_parts.append(pytesseract.image_to_string(image))
    return '\n'.join(text_parts).strip()


def read_image_bytes(file_bytes):
    image = Image.open(io.BytesIO(file_bytes))
    return pytesseract.image_to_string(image).strip()


def extract_text_from_uploaded_file(file_obj):
    if file_obj is None:
        raise ValueError('No file uploaded.')

    filename = getattr(file_obj, 'name', None)
    if not validate_extension_only(filename):
        raise ValueError('Unsupported file extension.')

    ext = get_extension_from_name(filename)
    file_bytes = file_obj.read()

    if ext == 'txt':
        return read_txt_bytes(file_bytes)
    if ext == 'docx':
        return read_docx_bytes(file_bytes)
    if ext == 'pdf':
        return read_pdf_bytes(file_bytes)
    if ext in {'png', 'jpg', 'jpeg'}:
        return read_image_bytes(file_bytes)

    raise ValueError('Unsupported file extension.')


## Question extraction and module matching

In [ ]:
def clean_text(text):
    text = text.replace('\r', '\n')
    text = re.sub(r'\n+', '\n', text)
    text = re.sub(r'[ \t]+', ' ', text)
    return text.strip()


def split_questions(text):
    text = clean_text(text)
    pattern = r'(?i)(?=\n?\s*(?:question\s*\d+|q\s*\d+|\d+\s*[\.\)]))'
    parts = re.split(pattern, '\n' + text)

    questions = []
    for part in parts:
        part = part.strip()
        if part and len(part.split()) >= 3:
            questions.append(part)

    if len(questions) <= 1:
        fallback_parts = [p.strip() for p in text.split('\n') if p.strip()]
        questions = [p for p in fallback_parts if len(p.split()) >= 4]

    return questions


def extract_module_items(module_text):
    module_text = clean_text(module_text)
    lines = [line.strip('-• \t') for line in module_text.split('\n')]
    lines = [line for line in lines if len(line) > 3]

    keywords = [
        'clo', 'course learning outcome', 'learning outcome',
        'topic', 'module', 'unit', 'chapter', 'outcome', 'objective'
    ]

    items = []
    for line in lines:
        low = line.lower()
        if any(k in low for k in keywords):
            items.append(line)
        elif len(line.split()) >= 3:
            items.append(line)

    seen = set()
    unique_items = []
    for item in items:
        key = item.lower()
        if key not in seen:
            seen.add(key)
            unique_items.append(item)

    return unique_items


def sentence_similarity(a, b):
    return SequenceMatcher(None, a.lower(), b.lower()).ratio()


def item_is_covered(module_item, questions, threshold=0.35):
    best_score = 0.0
    best_question = None
    for question in questions:
        score = sentence_similarity(module_item, question)
        if score > best_score:
            best_score = score
            best_question = question
    return best_score >= threshold, best_score, best_question


def evaluate_match(questions, module_items, item_threshold=0.35, coverage_threshold=0.60):
    if not module_items:
        return {
            'match': False,
            'coverage_ratio': 0.0,
            'missing_items': ['No module items were detected.'],
            'covered_items': []
        }

    covered_items = []
    missing_items = []

    for item in module_items:
        covered, score, matched_question = item_is_covered(
            item, questions, threshold=item_threshold
        )
        info = {
            'module_item': item,
            'score': round(score, 3),
            'matched_question': matched_question
        }
        if covered:
            covered_items.append(info)
        else:
            missing_items.append(info)

    coverage_ratio = len(covered_items) / len(module_items)
    is_match = coverage_ratio >= coverage_threshold

    return {
        'match': is_match,
        'coverage_ratio': round(coverage_ratio, 3),
        'missing_items': missing_items,
        'covered_items': covered_items
    }


def predict_bloom_levels(model, questions):
    if not questions:
        return []
    preds = model.predict(questions)
    return [{'question': q, 'predicted_bloom': p} for q, p in zip(questions, preds)]


## Test the pipeline with two files

In [ ]:
model, label_classes, saved_accuracy = load_pipeline_from_h5(MODEL_PATH)
print('Loaded model accuracy:', saved_accuracy)
print('Labels:', label_classes)


In [ ]:
# Optional quick test:
# from google.colab import files
# uploaded = files.upload()
#
# After upload, open files and test like this:
# with open('module.pdf', 'rb') as f:
#     module_text = extract_text_from_uploaded_file(f)
# with open('paper.pdf', 'rb') as f:
#     paper_text = extract_text_from_uploaded_file(f)
#
# questions = split_questions(paper_text)
# module_items = extract_module_items(module_text)
# bloom_results = predict_bloom_levels(model, questions)
# match_result = evaluate_match(questions, module_items)
#
# print(match_result)
# print(bloom_results[:3])


## Gradio interface with exactly two inputs

In [ ]:
import gradio as gr


def format_results(match_result, bloom_results):
    lines = []
    lines.append(f"Matched: {'YES' if match_result['match'] else 'NO'}")
    lines.append(f"Coverage Ratio: {match_result['coverage_ratio']}")
    lines.append('')

    lines.append('Predicted Bloom Levels:')
    if bloom_results:
        for i, item in enumerate(bloom_results, start=1):
            lines.append(f"{i}. {item['predicted_bloom']} -> {item['question']}")
    else:
        lines.append('No questions detected.')

    lines.append('')
    lines.append('Missing / Not Included Items:')
    if match_result['missing_items']:
        for item in match_result['missing_items']:
            if isinstance(item, dict):
                lines.append(f"- {item['module_item']} (score: {item['score']})")
            else:
                lines.append(f"- {item}")
    else:
        lines.append('None')

    return '\n'.join(lines)


def analyze_files(module_file, paper_file):
    if module_file is None:
        return 'Please upload the module file.'
    if paper_file is None:
        return 'Please upload the paper file.'

    if not validate_extension_only(module_file.name):
        return 'Module file extension is not supported.'
    if not validate_extension_only(paper_file.name):
        return 'Paper file extension is not supported.'

    with open(module_file.name, 'rb') as mf:
        module_text = extract_text_from_uploaded_file(mf)
    with open(paper_file.name, 'rb') as pf:
        paper_text = extract_text_from_uploaded_file(pf)

    model, _, _ = load_pipeline_from_h5(MODEL_PATH)

    questions = split_questions(paper_text)
    module_items = extract_module_items(module_text)
    bloom_results = predict_bloom_levels(model, questions)
    match_result = evaluate_match(questions, module_items)

    return format_results(match_result, bloom_results)


demo = gr.Interface(
    fn=analyze_files,
    inputs=[
        gr.File(label='Upload Module'),
        gr.File(label='Upload Paper')
    ],
    outputs=gr.Textbox(label='Result', lines=25),
    title='Exam Paper vs Module Matching',
    description='Upload only two files: one module and one paper.'
)

demo.launch(debug=True)


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://4fd457d2b941b6a984.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/gradio/queueing.py", line 759, in process_events
    response = await route_utils.call_process_api(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/gradio/route_utils.py", line 354, in call_process_api
    output = await app.get_blocks().process_api(
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/gradio/blocks.py", line 2191, in process_api
    result = await self.call_function(
             ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/gradio/blocks.py", line 1698, in call_function
    prediction = await anyio.to_thread.run_sync(  # type: ignore
                 ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/anyio/to_thread.py", line 63, in run_sync
    return await get_async_backend().run_sync_in_worker_thread(
           ^^^^^